# AuraGateway full-run environment qualification

This notebook is an execution surface, not authorization. It must not be run until the exact authorization, dataset manifest, and runtime factory binding have been merged and supplied. No benchmark trajectory is permitted.


In [ ]:
import hashlib
import importlib
import json
import os
import shutil
import stat
import sys
from datetime import UTC, datetime
from pathlib import Path

EXPECTED_REQUEST_SHA256 = (
    "7b0080429246f6def3c1ac28b8a677a2ed7e29ccf318690d9309ed98ff179ba0"
)
EXPECTED_RUNTIME_FACTORY = (
    "auragateway.local_abc."
    "full_abc_local_environment_qualification_kaggle_runtime_adapter:"
    "create_runtime_adapter"
)
KAGGLE_INPUT_ROOT = Path("/kaggle/input").resolve()
KAGGLE_WORKING_ROOT = Path("/kaggle/working").resolve()
MATERIALIZED_HARNESS_ROOT = KAGGLE_WORKING_ROOT / "auragateway_qualification_harness"


def _required_environment(name):
    value = os.environ.get(name)
    if value is None:
        raise RuntimeError(f"required environment binding is missing: {name}")
    return value


def _bounded_input_path(raw_path, *, expected_kind):
    path = Path(raw_path).resolve()
    if path != KAGGLE_INPUT_ROOT and KAGGLE_INPUT_ROOT not in path.parents:
        raise RuntimeError(f"{expected_kind} must remain under /kaggle/input")
    return path


def _file_sha256(path):
    if not path.is_file() or path.is_symlink():
        raise RuntimeError("expected one regular file")
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _canonical_sha256(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def _directory_sha256(root):
    entries = []
    total_bytes = 0
    for path in sorted(root.rglob("*"), key=lambda item: item.as_posix()):
        if path.is_symlink():
            raise RuntimeError("harness source contains a symbolic link")
        metadata = path.stat()
        if path.is_dir():
            continue
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError("harness source contains a non-regular member")
        total_bytes += metadata.st_size
        if total_bytes > 100 * 1024 * 1024:
            raise RuntimeError("harness source exceeds the bootstrap byte budget")
        entries.append(
            {
                "path": path.relative_to(root).as_posix(),
                "sha256": _file_sha256(path),
                "size_bytes": metadata.st_size,
            }
        )
        if len(entries) > 5_000:
            raise RuntimeError("harness source exceeds the bootstrap file budget")
    if not entries:
        raise RuntimeError("harness source is empty")
    return _canonical_sha256(entries)


def _parse_timestamp(raw_value):
    if not isinstance(raw_value, str):
        raise RuntimeError("authorization timestamp is invalid")
    normalized = raw_value.replace("Z", "+00:00")
    value = datetime.fromisoformat(normalized)
    if value.tzinfo is None or value.utcoffset() is None:
        raise RuntimeError("authorization timestamp must be timezone-aware")
    return value


authorization_path = _bounded_input_path(
    _required_environment("AURAGATEWAY_QUALIFICATION_AUTHORIZATION"),
    expected_kind="authorization",
)
manifest_path = _bounded_input_path(
    _required_environment("AURAGATEWAY_QUALIFICATION_DATASET_MANIFEST"),
    expected_kind="dataset manifest",
)
authorization = json.loads(authorization_path.read_text(encoding="utf-8"))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if authorization.get("decision") != "AUTHORIZED":
    raise RuntimeError("authorization decision is not AUTHORIZED")
if authorization.get("request_sha256") != EXPECTED_REQUEST_SHA256:
    raise RuntimeError("authorization does not bind the expected request")
if authorization.get("dataset_manifest_sha256") != _canonical_sha256(manifest):
    raise RuntimeError("authorization does not bind the supplied dataset manifest")
entries = manifest.get("entries")
expected_entry_shapes = (
    ("harness_source", "source_tree_directory"),
    ("model_artifacts", "hugging_face_snapshot_directory"),
    ("vllm_runtime", "python_wheelhouse_directory"),
)
if not isinstance(entries, list) or len(entries) != len(expected_entry_shapes):
    raise RuntimeError("dataset manifest entry set drifted")
for entry, expected_shape in zip(entries, expected_entry_shapes, strict=True):
    if not isinstance(entry, dict):
        raise RuntimeError("dataset manifest entry is invalid")
    observed_shape = (entry.get("role"), entry.get("artifact_format"))
    if observed_shape != expected_shape:
        raise RuntimeError("dataset manifest role or artifact format drifted")
    if expected_shape[0] != "vllm_runtime":
        _bounded_input_path(
            entry.get("mounted_path"),
            expected_kind=f"{expected_shape[0]} input",
        )
    elif entry.get("mounted_path") is not None:
        raise RuntimeError("vLLM runtime must use bounded output-directory discovery")
if any(
    (
        manifest.get("network_access_permitted") is not False,
        manifest.get("credentials_present") is not False,
        manifest.get("customer_data_present") is not False,
        manifest.get("hosted_provider_inputs_present") is not False,
    )
):
    raise RuntimeError("dataset manifest safety envelope drifted")
if any(
    (
        authorization.get("maximum_kaggle_sessions") != 1,
        authorization.get("maximum_model_requests") != 8,
        authorization.get("maximum_output_tokens_per_request") != 32,
        authorization.get("benchmark_trajectory_requests_permitted") != 0,
        authorization.get("customer_data_permitted") is not False,
        authorization.get("credentials_permitted") is not False,
        authorization.get("network_access_permitted") is not False,
        authorization.get("external_spend") != 0,
        authorization.get("measured_execution_authorized") is not False,
    )
):
    raise RuntimeError("authorization safety envelope drifted")
now = datetime.now(UTC)
issued_at = _parse_timestamp(authorization.get("issued_at"))
expires_at = _parse_timestamp(authorization.get("expires_at"))
if not issued_at <= now < expires_at:
    raise RuntimeError("authorization is outside its validity window")

harness_entries = [entry for entry in entries if entry.get("role") == "harness_source"]
if len(harness_entries) != 1:
    raise RuntimeError("dataset manifest must contain one harness_source entry")
harness_entry = harness_entries[0]
if harness_entry.get("artifact_format") != "source_tree_directory":
    raise RuntimeError("harness_source must use source_tree_directory")
mounted_repo_root = _bounded_input_path(
    harness_entry.get("mounted_path"),
    expected_kind="harness source",
)
if not mounted_repo_root.is_dir() or mounted_repo_root.is_symlink():
    raise RuntimeError("harness_source must resolve to one real directory")
harness_sha256 = harness_entry.get("sha256")
if _directory_sha256(mounted_repo_root) != harness_sha256:
    raise RuntimeError("harness_source identity does not match the manifest")
mounted_source_root = mounted_repo_root / "src"
required_harness_paths = (
    mounted_repo_root / "pyproject.toml",
    mounted_source_root / "auragateway",
    mounted_repo_root / "data/evals/benchmark/environment-qualification-v1/"
    "qualification_execution_request.json",
)
if not required_harness_paths[0].is_file():
    raise RuntimeError("harness_source does not contain pyproject.toml")
if not required_harness_paths[1].is_dir():
    raise RuntimeError("harness_source does not contain src/auragateway")
if not required_harness_paths[2].is_file():
    raise RuntimeError("harness_source does not contain the execution request")
if not KAGGLE_WORKING_ROOT.is_dir():
    raise RuntimeError("/kaggle/working is unavailable")
if MATERIALIZED_HARNESS_ROOT.exists():
    raise RuntimeError("writable harness destination already exists")
shutil.copytree(mounted_repo_root, MATERIALIZED_HARNESS_ROOT)
repo_root = MATERIALIZED_HARNESS_ROOT.resolve()
if _directory_sha256(repo_root) != harness_sha256:
    raise RuntimeError("writable harness copy identity drifted")
source_root = repo_root / "src"

runtime_factory = authorization.get("runtime_factory")
if not isinstance(runtime_factory, dict):
    raise RuntimeError("authorization runtime factory is invalid")
if runtime_factory.get("factory_path") != EXPECTED_RUNTIME_FACTORY:
    raise RuntimeError("authorization runtime factory path drifted")
adapter_relative_path = Path(runtime_factory.get("artifact_path", ""))
if adapter_relative_path.is_absolute() or ".." in adapter_relative_path.parts:
    raise RuntimeError("runtime adapter path must be repository-relative")
adapter_path = (repo_root / adapter_relative_path).resolve()
if adapter_path != repo_root and repo_root not in adapter_path.parents:
    raise RuntimeError("runtime adapter must remain inside harness_source")
if _file_sha256(adapter_path) != runtime_factory.get("artifact_sha256"):
    raise RuntimeError("runtime adapter identity does not match authorization")

os.environ["AURAGATEWAY_REPO_ROOT"] = str(repo_root)
sys.path.insert(0, str(source_root))

execution_module = importlib.import_module(
    "auragateway.local_abc.full_abc_local_environment_qualification_execution"
)
summary = execution_module.execute_from_environment()
summary